# Serial port reading

This project is a Jupyter notebook for reading, saving, and live-plotting data from a serial port (e.g., Arduino) using Python. The workflow is tailored for Windows and expects the `pyserial`, `numpy`, and `matplotlib` libraries.

## Project Overview
- **Main file:** `Lettura_serial.ipynb` (Jupyter Notebook)
- **Purpose:** Read numeric data from a serial port, save to text files, and optionally plot live data.
- **Typical hardware:** Arduino or similar microcontroller sending numeric data over serial.

## Key Patterns & Conventions
- **Serial Port Setup:**
  - Configure `porta_seriale` (e.g., `'COM3'`) and `baud_rate` to match your hardware.
  - Serial connection is managed with `serial.Serial` and closed in a `finally` block.
- **Data Validation:**
  - Use the `is_float(value)` function to ensure only numeric data is processed and saved.
  - Non-numeric lines are logged with a warning and ignored.
- **Data Storage:**
  - Data is appended to a list (`T` for temperature, `xdata` for time) and saved to `.txt` files using `numpy.savetxt`.
  - Output files are named (e.g., `dati_temperature.txt`, `Lettura_temp_py.txt`, `Prova_1.txt`) and include headers.
- **Live Plotting:**
  - Uses `matplotlib` in interactive mode (`plt.ion()`).
  - Graph updates in real time as new data arrives.
  - Error bars are added with a fixed error value (`err`).
- **Interrupt Handling:**
  - Graceful shutdown on `KeyboardInterrupt` (Ctrl+C), with serial port closure and user notification.

## Serial port reading
This code is about serial port reading and data saving on txt files. The microcontroller is to be wired up in a COM port.

In [ ]:
import serial
import time
import numpy as np
from matplotlib import pyplot as plt

# Initialization
porta_seriale = 'COM3'  # Serial port. It's to be configured by the user.
baud_rate = 115200  # Baud rate: many baud rate can be use, but the according with the arduino code running in the microcontreller is necessary.
nome_file = 'dati_temperature.txt'  # Raw temperature data file

ser = None 
T = []


# Function that verify if a value can be converted in a float. It is not necessary, but it prevents mistakes about serial output. Be sure that the output is a number.
def is_float(value):
    try:
        float(value)
        return True
    except ValueError:
        return False

# Serial reading
try:
    ser = serial.Serial(porta_seriale, baud_rate, timeout=1)
    print(f"[OK] Connection with {porta_seriale} at {baud_rate} baud")

    with open(nome_file, 'w') as file:
        try:
            while True:
                riga = ser.readline().decode('utf-8').strip()  # Reads the serial line and decodes it
                if riga and is_float(riga):  # Checks if the line is not empty and is a float
                    print(riga)
                    file.write(riga + '\n')  # Writes the line to the file
                    T.append(float(riga))  #  Append the float value to the list
                    file.flush() # Ensure data is written to file immediately
                elif riga:
                    print(f"[AVVISO] Dato non numerico ignorato: {riga}")  # Warn if the data is not a number
                time.sleep(0.1)
        except KeyboardInterrupt:
            print("\n[STOP] Interrotto dall'utente (CTRL+C)") # Graceful exit on user interrupt

except Exception as e:
    print(f"[ERRORE] Impossibile aprire la porta seriale: {e}") # Error if the serial port cannot be opened

# Cleanup
finally:
    if ser and ser.is_open:
        ser.close()
        print("[FINE] Porta seriale chiusa.")   #Colse the serial port

# Save data to a text file
if T:
    data = np.column_stack((T,))
    np.savetxt("C:/Users/valer/Desktop/Lavoro/Lettura_temp_py.txt", data, header='# T una per secondo', delimiter='   ') #The directory is to be configured by the user



[OK] Connessione avvenuta su COM3 a 115200 baud
24.44
24.44
24.44
24.44
24.44
24.44
24.44
24.44
24.37
24.37
24.37
24.37

[STOP] Interrotto dall'utente (CTRL+C)
[FINE] Porta seriale chiusa.


##  Serial port reading, saving and plotting
This code includes also live plotting of our serial data. It uses the 'TkAgg' beckend for interactive plots, opened on matplotlib.

In [ ]:
import matplotlib
matplotlib.use('TkAgg')  # Use TkAgg backend for interactive plots
import serial
import time
import numpy as np
import matplotlib.pyplot as plt

# Initialization
porta_seriale = 'COM3'  # The port is to be configured by the user
baud_rate = 115200
nome_file = 'dati_temperature.txt' # Raw temperature data file

ser = None
T = []
xdata = []
err = 0.03  # Assumed constant error for temperature measurements, this changes according to your sensor. We use the resolution of the sensor.

# Function that verify if a value can be converted in a float
def is_float(value):
    try:
        float(value)
        return True
    except ValueError:
        return False

# Setup plot
plt.ion()
fig, ax = plt.subplots()
ax.set_title("Temperatura in tempo reale")
ax.set_xlabel("Tempo [s]")
ax.set_ylabel("Temperatura [°C]")
line = None

inizio = time.time()

# Serial reading and plotting
try:
    ser = serial.Serial(porta_seriale, baud_rate, timeout=1)
    print(f"[OK] Connessione avvenuta su {porta_seriale} a {baud_rate} baud")

    with open(nome_file, 'w') as file:
        try:
            while True:
                riga = ser.readline().decode('utf-8').strip() # Reads the serial line and decodes it
                if riga and is_float(riga): # Checks if the line is not empty and is a float
                    temp = float(riga)
                    tempo = time.time() - inizio

                    print(f"{tempo:.1f}s: {temp} °C")
                    file.write(riga + '\n')
                    file.flush()

                    T.append(temp)  # Append the float value to the list
                    xdata.append(tempo) # Append the time value to the list

                    # Update plot
                    if line is not None:
                        line.remove()   # Remove the previous line
                    line = ax.errorbar(xdata, T, yerr=err, fmt='r.-', label="Temperatura ±0.06 °C") # Update the plot with error bars

                    ax.relim()  # Recalculate limits
                    ax.autoscale_view() # Autoscale
                    fig.canvas.draw()   # Draw the canvas
                    fig.canvas.flush_events()   # Flush events: necessary for interactive mode

                elif riga:
                    print(f"[AVVISO] Dato non numerico ignorato: {riga}") # Warn if the data is not a number
                time.sleep(0.1)

        except KeyboardInterrupt:
            print("\n[STOP] Interrotto dall'utente (CTRL+C)")   # Graceful exit on user interrupt

except Exception as e:
    print(f"[ERRORE] Impossibile aprire la porta seriale: {e}") # Error if the serial port cannot be opened

# Cleanup
finally:
    if ser and ser.is_open:
        ser.close()
        print("[FINE] Porta seriale chiusa.")

# Save data to a text file
if T:
    data = np.column_stack((xdata, T))
    np.savetxt("C:/Users/valer/Desktop/Lavoro/Prova_1.txt", data, header='# tempo[s]   temperatura[°C]', delimiter='   ')   #The saving directory is to be configured by the user. It saves time and temperature


[OK] Connessione avvenuta su COM3 a 115200 baud
0.5s: 26.99 °C
1.6s: 27.05 °C
2.8s: 27.02 °C
3.9s: 26.99 °C
5.1s: 26.99 °C
6.2s: 26.95 °C
7.4s: 26.95 °C
8.5s: 26.99 °C
9.7s: 26.95 °C
10.8s: 26.95 °C
12.0s: 26.92 °C
13.1s: 26.92 °C
14.3s: 26.92 °C
15.4s: 26.95 °C
16.6s: 26.92 °C
17.7s: 26.99 °C
18.9s: 29.36 °C
20.0s: 30.47 °C
21.2s: 30.98 °C
22.3s: 31.25 °C
23.5s: 31.25 °C
24.6s: 31.36 °C
25.8s: 31.42 °C
26.9s: 31.46 °C
28.1s: 31.59 °C
29.2s: 31.59 °C
30.4s: 31.69 °C
31.5s: 31.66 °C
32.7s: 31.73 °C
33.8s: 31.69 °C
35.0s: 31.76 °C
36.1s: 31.73 °C
37.3s: 31.8 °C
38.4s: 31.8 °C
39.6s: 31.8 °C
40.7s: 31.76 °C
41.9s: 31.8 °C
43.0s: 31.8 °C
44.2s: 31.86 °C
45.3s: 31.86 °C
46.5s: 31.52 °C
47.6s: 31.02 °C
48.8s: 30.58 °C
50.0s: 30.17 °C
51.1s: 29.83 °C
52.3s: 29.59 °C
53.4s: 29.29 °C
54.6s: 29.02 °C
55.7s: 28.88 °C
56.9s: 28.65 °C
58.0s: 28.44 °C
59.2s: 28.31 °C
60.3s: 28.17 °C
61.5s: 28.04 °C
62.6s: 27.97 °C
63.8s: 27.87 °C
64.9s: 27.8 °C
66.1s: 27.73 °C
67.2s: 27.66 °C
68.4s: 27.63 °C
69.5s: 